# Day 8 — Principles of Machine Learning
**30-Day ML & AI Mastery Plan · Shubham · Week 2 starts**

Week 1 built the foundations. Week 2 is classical machine learning, and it starts with the one idea every other day depends on:

> **A model is only useful if it works on data it has never seen.**

Scoring 100% on your training data is easy (just memorise it). The whole craft of ML is building models that **generalise**, and measuring that honestly. Today you'll learn how to split data, how to evaluate with cross-validation, how to recognise overfitting and underfitting, why they happen (bias and variance), and you'll build two classic classifiers, **k-NN** and a **linear classifier**, and watch their decision boundaries.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | Generalisation: why training accuracy lies | 15 min |
| 2 | Train / validation / test splits | 20 min |
| 3 | Cross-validation | 25 min |
| 4 | Overfitting vs underfitting | 20 min |
| 5 | The bias–variance trade-off, by simulation | 25 min |
| 6 | k-Nearest Neighbours from scratch | 30 min |
| 7 | Linear classifiers: a straight-line boundary | 20 min |
| **8** | **Live demo: decision boundaries as k changes (Iris and Wine)** | 35 min |
| 9 | Putting it together: tune, then test once | 20 min |
| 10 | Exercises | 30–60 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import (train_test_split, cross_val_score, KFold, StratifiedKFold,
                                     GridSearchCV, validation_curve)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(seed=8)

wine = load_wine()
X_wine, y_wine = wine.data, wine.target
print("Wine:", X_wine.shape, "| classes:", [str(n) for n in wine.target_names], "| counts:", np.bincount(y_wine))

---
# Part 1 — Generalisation: why training accuracy lies

Imagine a student who memorises last year's exam answers. They'd score 100% on last year's paper and fail this year's. A model can do exactly the same. The **1-nearest-neighbour** classifier ("predict the label of the single closest training point") memorises perfectly: every training point is its own nearest neighbour.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine)
memoriser = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_tr)
print(f"1-NN accuracy on the data it was trained on: {memoriser.score(X_tr, y_tr):.3f}")
print(f"1-NN accuracy on wines it has never seen     : {memoriser.score(X_te, y_te):.3f}")

100% on training data, much lower on new data. **Training accuracy measures memory, not skill.** Everything today is about measuring the second number honestly and making it as high as possible.

The gap between training and test performance is the **generalisation gap**. A big gap means the model learned details specific to the training rows (noise) instead of patterns that hold in general.

---
# Part 2 — Train, validation and test sets

| Set | Used for | Analogy |
|---|---|---|
| **Training** (~60–80%) | fitting the model's parameters | studying the textbook |
| **Validation** (~10–20%) | choosing between models and settings (hyperparameters) | mock exams |
| **Test** (~10–20%) | one final, honest estimate, **used once** | the real exam |

Why a separate validation set? If you try 50 settings and pick the one with the best **test** score, that score is now optimistic: you've tuned to the test set by trial and error. The test set must stay untouched until the very end.

**Rules for splitting:**
- **Stratify** classification labels so each set keeps the same class proportions (Day 4).
- **Time series:** train on the past, validate and test on the future. Never shuffle.
- **Groups** (several rows per customer or patient): keep each group in one set only, or the model meets "known" people at test time.
- Fix the **random seed** so results are reproducible.

In [ ]:
# Two calls to train_test_split give 60% / 20% / 20%
X_temp, X_test, y_temp, y_test = train_test_split(X_wine, y_wine, test_size=0.20, random_state=8, stratify=y_wine)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=8, stratify=y_temp)  # 0.25 x 80% = 20%

for name, labels in [("all", y_wine), ("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:<11} n = {len(labels):>3}   class shares = {np.round(np.bincount(labels) / len(labels), 2)}")

In [ ]:
# Choose k on the VALIDATION set, then report the TEST score once
val_scores = {}
for k in [1, 3, 5, 7, 9, 15, 25]:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    val_scores[k] = model.score(X_val, y_val)
best_k = max(val_scores, key=val_scores.get)
print("validation accuracy by k:", {k: round(v, 3) for k, v in val_scores.items()})
print("chosen k:", best_k)

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_temp, y_temp)  # refit on train + val
print(f"final test accuracy (reported once): {final.score(X_test, y_test):.3f}")

Notice that several values of k tie on the validation set. With only 36 validation wines, one wine = 2.8 percentage points, so small differences mean little. That instability is exactly the problem cross-validation solves.

---
# Part 3 — Cross-validation: using every row for validation

A single split is a single roll of the dice. How much does the score depend on *which* rows ended up in the test set?

In [ ]:
scores = []
for seed in range(100):
    a, b, c, d = train_test_split(X_wine, y_wine, test_size=0.3, random_state=seed, stratify=y_wine)
    scores.append(make_pipeline(StandardScaler(), KNeighborsClassifier(5)).fit(a, c).score(b, d))
scores = np.array(scores)
print(f"same model, 100 different random splits: accuracy from {scores.min():.3f} to {scores.max():.3f} (std {scores.std():.3f})")

The same model looks anywhere from mediocre to perfect depending on the split. **k-fold cross-validation** fixes this:

1. Split the data into k equal **folds** (usually 5 or 10).
2. Train on k − 1 folds, validate on the remaining one.
3. Repeat k times so every fold is the validation set once.
4. Report the **mean** score (and its spread).

Every row is used for validation exactly once and for training k − 1 times. **Stratified** k-fold keeps class proportions in each fold (scikit-learn does this by default for classifiers).

In [ ]:
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
fold_scores = cross_val_score(model, X_wine, y_wine, cv=cv)
print("accuracy per fold:", fold_scores.round(3))
print(f"5-fold CV accuracy: {fold_scores.mean():.3f} ± {fold_scores.std():.3f}")

# What one fold looks like
for i, (train_idx, val_idx) in enumerate(cv.split(X_wine, y_wine)):
    if i == 0:
        print(f"fold 1: train on {len(train_idx)} wines, validate on {len(val_idx)}; validation class counts {np.bincount(y_wine[val_idx])}")

Because the scaler lives **inside** the pipeline, `cross_val_score` refits it on each training fold only. Scaling the whole dataset before cross-validation would leak the validation folds' statistics into training (Day 4's rule, again).

### Other splitters for real-world data
| Data | Splitter | Why |
|---|---|---|
| ordinary classification | `StratifiedKFold` | keeps class balance |
| several rows per person / customer | `GroupKFold` | a person never appears in both train and validation |
| time series | `TimeSeriesSplit` | always trains on the past, validates on the future |
| tiny datasets | `LeaveOneOut` or repeated k-fold | uses data efficiently |

---
# Part 4 — Overfitting vs underfitting

| | Underfitting | Good fit | Overfitting |
|---|---|---|---|
| What happens | model too simple to capture the pattern | captures the pattern | memorises noise as if it were pattern |
| Training error | high | low | very low |
| Validation error | high | low | **high** |
| Fix | more features, more flexible model, less regularisation | | more data, simpler model, regularisation, early stopping |

The clearest way to see it: fit polynomials of increasing degree to 20 noisy points from a smooth curve.

In [ ]:
def true_fn(x):
    return np.sin(2 * np.pi * x)

x_train = np.sort(rng.uniform(0, 1, 20)); y_train_p = true_fn(x_train) + rng.normal(0, 0.25, 20)
x_val = np.sort(rng.uniform(0, 1, 200)); y_val_p = true_fn(x_val) + rng.normal(0, 0.25, 200)
grid = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, degree, label in zip(axes, [1, 4, 15], ["underfit", "good fit", "overfit"]):
    poly = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_train[:, None], y_train_p)
    tr = mean_squared_error(y_train_p, poly.predict(x_train[:, None]))
    va = mean_squared_error(y_val_p, poly.predict(x_val[:, None]))
    ax.scatter(x_train, y_train_p, color="black", s=20, label="training points")
    ax.plot(grid, true_fn(grid), "g--", label="true pattern")
    ax.plot(grid, poly.predict(grid[:, None]), "r", label=f"degree {degree}")
    ax.set_ylim(-2, 2); ax.legend(loc="lower left", fontsize=8)
    ax.set_title(f"degree {degree}: {label}\ntrain MSE {tr:.3f} | validation MSE {va:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
degrees = range(1, 16)
train_err, val_err = [], []
for dg in degrees:
    poly = make_pipeline(PolynomialFeatures(dg), LinearRegression()).fit(x_train[:, None], y_train_p)
    train_err.append(mean_squared_error(y_train_p, poly.predict(x_train[:, None])))
    val_err.append(mean_squared_error(y_val_p, poly.predict(x_val[:, None])))

plt.figure(figsize=(8, 3.8))
plt.plot(degrees, train_err, "o-", label="training error"); plt.plot(degrees, val_err, "o-", label="validation error")
plt.yscale("log"); plt.xlabel("polynomial degree (model complexity)"); plt.ylabel("MSE (log)")
best = degrees[int(np.argmin(val_err))]
plt.axvline(best, color="grey", ls="--"); plt.title(f"Training error keeps falling; validation error is U-shaped (best degree {best})")
plt.legend(); plt.show()

This U-shaped validation curve is the most important picture in ML. **Training error always falls as models get more complex; validation error falls, then rises.** The sweet spot is the bottom of the U, and you find it with validation data, never with training data.

---
# Part 5 — The bias–variance trade-off

Why does the U happen? Imagine retraining the same kind of model on many different training sets drawn from the same world. For a given input x, the expected squared error splits into three parts:

$$E\big[(y - \hat f(x))^2\big] = \underbrace{\big(E[\hat f(x)] - f(x)\big)^2}_{\text{bias}^2} + \underbrace{E\big[(\hat f(x) - E[\hat f(x)])^2\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{noise}}$$

- **Bias**: how far the *average* model is from the truth. Simple models are systematically wrong (a straight line can't follow a wave).
- **Variance**: how much the model *changes* between training sets. Complex models chase each set's particular noise.
- **Noise**: randomness in the data itself; no model can remove it.

Dartboard analogy: high bias = darts clustered but off-centre; high variance = darts scattered all over; the goal is clustered *and* centred. We can measure both directly by simulation:

In [ ]:
def bias_variance(degree, n_datasets=500, n_points=25, noise=0.25):
    x_eval = np.linspace(0.1, 0.9, 50)          # judge the fit away from the extreme edges
    preds = np.empty((n_datasets, x_eval.size))
    for i in range(n_datasets):
        xs = rng.uniform(0, 1, n_points)
        ys = true_fn(xs) + rng.normal(0, noise, n_points)
        m = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(xs[:, None], ys)
        preds[i] = m.predict(x_eval[:, None])
    bias_sq = np.mean((preds.mean(axis=0) - true_fn(x_eval)) ** 2)
    variance = np.mean(preds.var(axis=0))
    return bias_sq, variance, preds, x_eval

rows = []
for dg in [1, 3, 5, 7, 9, 11]:
    b2, v, _, _ = bias_variance(dg)
    rows.append({"degree": dg, "bias²": b2, "variance": v, "bias² + variance + noise": b2 + v + 0.25**2})
bv = pd.DataFrame(rows).set_index("degree")
bv.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, dg in zip(axes[:2], [1, 11]):
    _, _, preds, x_eval = bias_variance(dg, n_datasets=40)
    for p in preds: ax.plot(x_eval, p, color="tab:red", alpha=0.15)
    ax.plot(x_eval, true_fn(x_eval), "g--", lw=2, label="truth")
    ax.plot(x_eval, preds.mean(axis=0), "k", lw=2, label="average model")
    ax.set_ylim(-2, 2); ax.legend(loc="lower left"); ax.set_title(f"degree {dg}: 40 models from 40 training sets")
axes[2].plot(bv.index, bv["bias²"], "o-", label="bias²"); axes[2].plot(bv.index, bv["variance"], "o-", label="variance")
axes[2].plot(bv.index, bv["bias² + variance + noise"], "o-", label="total expected error")
axes[2].set_yscale("log"); axes[2].set(xlabel="degree", title="Bias falls, variance rises"); axes[2].legend()
plt.tight_layout(); plt.show()

Degree 1: every model is nearly the same line (low variance) but they're all wrong in the same way (high bias). Degree 11: the *average* model is almost exactly the truth (bias near 0), but individual models swing wildly from one training set to the next (variance 30× higher). The total error is smallest at degree 3, in between.

**Levers you'll use all through Week 2:**
| To reduce bias | To reduce variance |
|---|---|
| more flexible model, more features | more training data |
| less regularisation | more regularisation (Day 9: Ridge, Lasso) |
| train longer (neural nets) | simpler model, feature selection |
| boosting (Day 11) | bagging / Random Forests (Day 11), averaging many models |

---
# Part 6 — k-Nearest Neighbours, from scratch

The simplest classifier there is: **to classify a new point, find the k training points closest to it and take a vote.**

- No training step at all: it just stores the data ("lazy learning"). All the work happens at prediction time.
- **k controls the bias–variance trade-off**: k = 1 copies the single nearest point (low bias, high variance); a huge k votes over most of the dataset and predicts the majority class everywhere (high bias, low variance).
- It depends entirely on **distance** (Day 5), so features must be **scaled** (Day 4).

In [ ]:
from collections import Counter

class MyKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):                      # "training" = remembering the data
        self.X, self.y = np.asarray(X), np.asarray(y)
        return self

    def predict_one(self, x):
        distances = np.linalg.norm(self.X - x, axis=1)        # Euclidean distance to every training point
        nearest = np.argsort(distances)[:self.k]              # indices of the k closest
        return Counter(self.y[nearest]).most_common(1)[0][0]  # majority vote

    def predict(self, X):
        return np.array([self.predict_one(x) for x in np.asarray(X)])

scaler = StandardScaler().fit(X_tr)
mine = MyKNN(k=5).fit(scaler.transform(X_tr), y_tr)
theirs = KNeighborsClassifier(n_neighbors=5).fit(scaler.transform(X_tr), y_tr)
p_mine, p_theirs = mine.predict(scaler.transform(X_te)), theirs.predict(scaler.transform(X_te))
print(f"my k-NN accuracy     : {accuracy_score(y_te, p_mine):.3f}")
print(f"sklearn k-NN accuracy: {accuracy_score(y_te, p_theirs):.3f}")
print("identical predictions:", (p_mine == p_theirs).mean() == 1.0)

In [ ]:
# Scaling matters enormously for k-NN: wine features range from ~0.1 (hue) to ~1,700 (proline)
print(pd.DataFrame(X_wine, columns=wine.feature_names)[["proline", "magnesium", "hue", "nonflavanoid_phenols"]].describe().loc[["min", "max"]])
for name, m in [("raw features", KNeighborsClassifier(5)), ("standardised", make_pipeline(StandardScaler(), KNeighborsClassifier(5)))]:
    s = cross_val_score(m, X_wine, y_wine, cv=cv)
    print(f"k-NN with {name:<13}: 5-fold CV accuracy {s.mean():.3f}")

Without scaling, distance is dominated by `proline` (values in the hundreds and thousands), so k-NN effectively uses one feature out of thirteen.

### The curse of dimensionality
In high dimensions, points become nearly **equally far** from each other, so "nearest" stops meaning much. That's why k-NN works best with a modest number of meaningful, scaled features.

In [ ]:
for d in [2, 10, 100, 1000]:
    pts = rng.uniform(size=(500, d))
    dists = np.linalg.norm(pts[1:] - pts[0], axis=1)
    print(f"{d:>4} dimensions: nearest / farthest distance ratio = {dists.min() / dists.max():.2f}")

In 2-D the nearest point is many times closer than the farthest; in 1,000-D they're almost the same distance away.

---
# Part 7 — Linear classifiers

A linear classifier computes a score **z = w · x + b** (a dot product, Day 5) and predicts one class when z > 0 and the other when z < 0. The **decision boundary** is where z = 0:

- in 2-D, a straight **line**
- in 3-D, a flat **plane**
- in more dimensions, a **hyperplane**

**Logistic regression** is the most common linear classifier: it turns z into a probability with the sigmoid (Day 6) and learns w by minimising log loss with gradient descent. Others: the perceptron, linear SVM (Day 10). For more than two classes, scikit-learn fits one score per class and predicts the largest.

| | k-NN | Linear classifier |
|---|---|---|
| Boundary shape | any shape, follows the data | straight line / plane |
| Training | none (stores data) | learns w and b |
| Prediction speed | slow (distance to every training point) | instant (one dot product) |
| Interpretability | "these 5 similar cases" | a weight per feature |
| Typical weakness | high variance at small k; needs scaling; curse of dimensionality | high bias when classes aren't linearly separable |

In [ ]:
# Two features only, so we can see the boundary
X2 = StandardScaler().fit_transform(X_wine[:, [0, 9]])              # alcohol, colour intensity
y2 = (y_wine == 0).astype(int)                                       # class 0 vs the rest
logreg = LogisticRegression().fit(X2, y2)
w, b = logreg.coef_[0], logreg.intercept_[0]
print(f"learned: z = {w[0]:.2f} * alcohol + {w[1]:.2f} * colour + {b:.2f}")

xx = np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 100)
boundary = -(w[0] * xx + b) / w[1]                                    # solve w0*x + w1*y + b = 0 for y
plt.figure(figsize=(6.5, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm", edgecolor="k", s=30)
plt.plot(xx, boundary, "k--", lw=2, label="decision boundary: z = 0")
plt.ylim(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5)
plt.xlabel("alcohol (standardised)"); plt.ylabel("colour intensity (standardised)")
plt.title(f"Linear classifier: class 0 vs rest (train accuracy {logreg.score(X2, y2):.2f})"); plt.legend(); plt.show()

---
# Part 8 — LIVE DEMO: decision boundaries as k changes

We colour every point on a fine grid by the class the model predicts there. The borders between colours are the **decision boundaries**. Using two features lets us see them directly.

In [ ]:
def plot_boundary(model, X, y, ax, title, class_names):
    pad = 0.5
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, 300),
                         np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis", levels=np.arange(len(class_names) + 1) - 0.5)
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", edgecolor="k", s=22)
    ax.set_title(title, fontsize=10)
    return scatter

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data[:, [0, 1]])        # sepal length & width: classes overlap
y_iris = iris.target
ks = [1, 5, 15, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, k in zip(axes.ravel(), ks):
    m = KNeighborsClassifier(n_neighbors=k).fit(X_iris, y_iris)
    cv_acc = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_iris, y_iris, cv=StratifiedKFold(5, shuffle=True, random_state=8)).mean()
    plot_boundary(m, X_iris, y_iris, ax, f"k = {k}: train {m.score(X_iris, y_iris):.2f} | CV {cv_acc:.2f}", iris.target_names)
lin = LogisticRegression(max_iter=1000).fit(X_iris, y_iris)
lin_cv = cross_val_score(LogisticRegression(max_iter=1000), X_iris, y_iris, cv=StratifiedKFold(5, shuffle=True, random_state=8)).mean()
sc = plot_boundary(lin, X_iris, y_iris, axes[1, 2], f"linear (logistic regression): train {lin.score(X_iris, y_iris):.2f} | CV {lin_cv:.2f}", iris.target_names)
for ax in axes.ravel(): ax.set(xlabel="sepal length (std)", ylabel="sepal width (std)")
fig.legend(sc.legend_elements()[0], iris.target_names, loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.show()

Read the panels left to right:
- **k = 1**: islands and jagged borders around single flowers: the boundary bends to fit individual points. It has the highest training accuracy (0.92) and the lowest CV accuracy (0.71): the biggest gap, which is **overfitting (high variance)**. (Training accuracy isn't 1.00 because some flowers have identical sepal measurements but different species: no model can separate those.)
- **k = 5 and 15**: smoother borders that follow the main groups.
- **k = 50 and 100**: very smooth, blunt borders. Push k all the way to 150 (the whole dataset) and every point would vote for the same class: **underfitting (high bias)**.
- **Linear**: straight borders. On these two features it has the best CV score of all (0.81).

Notice that k-NN's CV scores barely move between k = 5 and k = 100 (0.75–0.78). Versicolor and virginica overlap so heavily on sepal measurements that no boundary can separate them well, and with 150 flowers those small differences are within the noise. The lesson is the **shape** of the boundaries and the **train–CV gap**, not the third decimal.

Now the same on the Wine data, plus the full **validation curve** for k:

In [ ]:
X_w2 = StandardScaler().fit_transform(X_wine[:, [0, 9]])             # alcohol, colour intensity
fig, axes = plt.subplots(1, 4, figsize=(21, 4.6))
for ax, k in zip(axes[:3], [1, 10, 60]):
    m = KNeighborsClassifier(n_neighbors=k).fit(X_w2, y_wine)
    cv_acc = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_w2, y_wine, cv=cv).mean()
    plot_boundary(m, X_w2, y_wine, ax, f"Wine, k = {k}: train {m.score(X_w2, y_wine):.2f} | CV {cv_acc:.2f}", wine.target_names)
    ax.set(xlabel="alcohol (std)", ylabel="colour intensity (std)")

k_range = np.arange(1, 61, 2)
train_s, val_s = validation_curve(make_pipeline(StandardScaler(), KNeighborsClassifier()), X_wine, y_wine,
                                  param_name="kneighborsclassifier__n_neighbors", param_range=k_range, cv=cv)
axes[3].plot(k_range, train_s.mean(axis=1), label="training accuracy")
axes[3].plot(k_range, val_s.mean(axis=1), label="cross-validated accuracy")
axes[3].fill_between(k_range, val_s.mean(axis=1) - val_s.std(axis=1), val_s.mean(axis=1) + val_s.std(axis=1), alpha=0.2)
axes[3].set(xlabel="k  (← more complex | simpler →)", ylabel="accuracy", title="All 13 features: validation curve for k")
axes[3].legend(); plt.tight_layout(); plt.show()
best_k_cv = k_range[np.argmax(val_s.mean(axis=1))]
print(f"best k by cross-validation (13 features): {best_k_cv}, CV accuracy {val_s.mean(axis=1).max():.3f}")

On two Wine features the story is textbook: k = 1 scores 1.00 on training but 0.79 in CV (overfit), k = 10 scores 0.84 on both (generalising), and k = 60 drops to 0.78 / 0.75 (underfit: the boundaries can no longer follow the classes).

The validation curve (right) uses all 13 features. Training accuracy starts at 1.00 for k = 1 and drops straight away; cross-validated accuracy starts lowest at k = 1, peaks around k = 13, then drifts down for large k. The shaded band (± one standard deviation across the 5 folds) is wider than the differences between most k values, a reminder that with 178 wines, many settings are statistically tied.

---
# Part 9 — Putting it together: tune with CV, test once

The professional workflow, which you'll repeat every day this week:
1. Hold out a **test set** first and don't touch it.
2. Put preprocessing and model in a **Pipeline**.
3. Tune hyperparameters with **cross-validation on the training set** (`GridSearchCV`).
4. Refit the best model on the full training set (GridSearchCV does it automatically).
5. Evaluate **once** on the test set and report that number.

In [ ]:
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=8, stratify=y_wine)

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
param_grid = {
    "kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31],
    "kneighborsclassifier__weights": ["uniform", "distance"],   # distance: closer neighbours get bigger votes
    "kneighborsclassifier__p": [1, 2],                          # 1 = Manhattan, 2 = Euclidean (Day 5)
}
search = GridSearchCV(pipe, param_grid, cv=StratifiedKFold(5, shuffle=True, random_state=8))
search.fit(X_train_full, y_train_full)
print("best settings:", {k.split("__")[1]: v for k, v in search.best_params_.items()})
print(f"best CV accuracy (training data): {search.best_score_:.3f}")
print(f"test accuracy (reported once)   : {search.score(X_test_final, y_test_final):.3f}")

lin_search = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                          {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}, cv=StratifiedKFold(5, shuffle=True, random_state=8))
lin_search.fit(X_train_full, y_train_full)
print(f"\nlinear classifier: best C = {lin_search.best_params_['logisticregression__C']}, CV {lin_search.best_score_:.3f}, test {lin_search.score(X_test_final, y_test_final):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(search, X_test_final, y_test_final, display_labels=wine.target_names, ax=axes[0], colorbar=False)
axes[0].set_title("Tuned k-NN on the test set")
ConfusionMatrixDisplay.from_estimator(lin_search, X_test_final, y_test_final, display_labels=wine.target_names, ax=axes[1], colorbar=False)
axes[1].set_title("Tuned linear classifier on the test set")
plt.tight_layout(); plt.show()

With all 13 features standardised, both models do well: the wine classes are close to **linearly separable** in 13 dimensions, even though they overlap in any two. The linear classifier wins here, and its test score (0.978) matches its CV estimate (0.985). k-NN's test score (0.911) is below its CV estimate (0.978).

Is that drop alarming? With 45 test wines, each mistake costs 2.2 points and the standard error of an accuracy near 93% is about 4 points (Day 4), so a 3-wine difference is within the noise of a small test set. That's why we choose models by **cross-validation** and treat a small test set's score as an estimate with an error bar. To compare models more reliably, use repeated cross-validation.

---
# Part 10 — Exercises
Paste your code in chat for review.

**Ex 1 — Split stability.** For 1-NN and 15-NN (with scaling) on Wine, run 100 random 70/30 splits and plot both accuracy distributions as histograms. Which model's score varies more, and how does that connect to variance?

**Ex 2 — Your own k-NN, improved.** Add a `weights="distance"` option to `MyKNN` (votes weighted by 1/distance) and a `predict_proba` method (share of neighbour votes per class). Check both against scikit-learn.

**Ex 3 — Learning curve.** Use `sklearn.model_selection.learning_curve` with a scaled 5-NN on Wine to plot training and CV accuracy against training-set size (10% to 100%). Does more data still help at the end? What would the curve look like for a high-bias model?

**Ex 4 — Leakage experiment.** Compare 5-fold CV accuracy of k-NN on Wine when (a) the scaler is inside the pipeline vs (b) `StandardScaler().fit_transform(X)` is applied to all data first. Is there a difference here? Explain why (b) is still wrong and when it would matter more.

**Stretch — Iris boundaries with your own k-NN.** Recreate the k = 1 and k = 15 boundary plots using `MyKNN` instead of scikit-learn.

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Stretch

---
### Day 8 recap
- **Generalisation** is the goal; training accuracy measures memory (1-NN scores 100% on its own training data).
- **Train / validation / test:** fit / choose / report once. Stratify; respect time order and groups.
- **Cross-validation:** k folds, each used once for validation; report mean ± std. Single splits are noisy. Keep preprocessing inside the pipeline.
- **Underfitting** = high error everywhere (high bias); **overfitting** = low training error, high validation error (high variance). Validation error is U-shaped in model complexity.
- **Bias–variance:** expected error = bias² + variance + noise. More data and regularisation cut variance; more flexible models cut bias.
- **k-NN:** store the data, vote among the k closest. Small k → jagged boundaries and overfitting; large k → over-smoothed and underfitting. Always scale; beware the curse of dimensionality.
- **Linear classifiers:** z = w·x + b, boundary where z = 0 (a line or hyperplane). Fast and interpretable, but biased when classes need curved borders.
- **Workflow:** hold out test → pipeline → GridSearchCV on train → test once.
- **Tomorrow (Day 9): regression and classification in depth**: linear regression (normal equation vs gradient descent), Ridge and Lasso, logistic regression, and every metric: MAE, RMSE, R², precision, recall, F1, ROC-AUC.